# Lecture 19B Worksheet: Entropy by Hand, Ensembles and XGBoost
**NAVTTC / NIAI AI/ML Trainer Cohort** — works in Google Colab. Fill every `___` blank and run top to bottom; check lines print `OK`.

In [1]:
import numpy as np, pandas as pd
from math import log2
pd.set_option("display.width", 120)

# Our invented 10-order table (lecture slide 3)
orders = pd.DataFrame({
    "rush":   [0,0,1,1,0,1,0,1,0,0],
    "rain":   [1,0,1,0,1,0,1,1,1,1],
    "coupon": [1,1,0,0,0,1,0,1,1,1],
    "late":   [0,1,1,1,0,1,0,1,0,0]}, index=range(1, 11))
print(orders)
print("late orders:", orders.late.sum(), "of", len(orders))

    rush  rain  coupon  late
1      0     1       1     0
2      0     0       1     1
3      1     1       0     1
4      1     0       0     1
5      0     1       0     0
6      1     0       1     1
7      0     1       0     0
8      1     1       1     1
9      0     1       1     0
10     0     1       1     0
late orders: 5 of 10


## Task 1: The entropy function
`H(p) = -p*log2(p) - (1-p)*log2(1-p)`, with `H(0) = H(1) = 0`. Fill in the blanks.

In [2]:
def H(p):
    if p == 0 or p == 1:
        return 0
    return -p*log2(p) - (1-p)*log2(1-p)

for p in [0, 1/6, 0.2, 2/7, 0.4, 0.5, 0.9, 1]:
    print(round(p, 3), round(H(p), 3))
assert H(0.5) == 1 and H(0) == 0 and abs(H(2/7) - 0.863) < 0.001; print("OK")

0 0
0.167 0.65
0.2 0.722
0.286 0.863
0.4 0.971
0.5 1.0
0.9 0.469
1 0
OK


## Task 2: Information gain of one feature
`gain = H(parent) - (w_left*H(p_left) + w_right*H(p_right))`. Split the orders on a yes/no feature (1 = left, 0 = right).

In [4]:
def info_gain(df, feature, target="late"):
    p_parent = df[target].mean()
    left, right = df[df[feature] == 1], df[df[feature] == 0]
    w_left, w_right = len(left)/len(df), len(right)/len(df)
    h_left  = H(left[target].mean())  if len(left)  else 0
    h_right = H(right[target].mean()) if len(right) else 0
    return H(p_parent) - (w_left*h_left + w_right*h_right)

for f in ["rush", "rain", "coupon"]:
    print(f, round(info_gain(orders, f), 3))
assert abs(info_gain(orders, "rush") - 0.610) < 0.001 and abs(info_gain(orders, "rain") - 0.396) < 0.001 and abs(info_gain(orders, "coupon")) < 1e-9; print("OK")

rush 0.61
rain 0.396
coupon 0.0
OK


## Task 3: Pick the best feature, then recurse
Write `best_feature` (the feature with the biggest gain), then build the tree recursively. A node is a leaf when all its orders have the same answer, or no feature is left.

In [5]:
def best_feature(df, features):
    return max(features, key=lambda f: info_gain(df, f))

def build_tree(df, features, depth=0):
    y = df.late
    if y.nunique() == 1 or not features:
        print("  "*depth + f"LEAF: {'late' if y.mean() >= 0.5 else 'on time'}  ({len(df)} orders, {y.sum()} late)")
        return
    f = best_feature(df, features)
    print("  "*depth + f"ask: {f}?  (gain {info_gain(df, f):.3f}, {len(df)} orders, {y.sum()} late)")
    for value, name in [(1, "yes"), (0, "no")]:
        print("  "*depth + f" {name}:")
        build_tree(df[df[f] == value], [g for g in features if g != f], depth + 1)

build_tree(orders, ["rush", "rain", "coupon"])
assert best_feature(orders, ["rush", "rain", "coupon"]) == "rush"; print("OK")

ask: rush?  (gain 0.610, 10 orders, 5 late)
 yes:
  LEAF: late  (4 orders, 4 late)
 no:
  ask: rain?  (gain 0.650, 6 orders, 1 late)
   yes:
    LEAF: on time  (5 orders, 0 late)
   no:
    LEAF: late  (1 orders, 1 late)
OK


## Task 4: Compare with scikit-learn
Train sklearn's `DecisionTreeClassifier(criterion="entropy")` on the same 10 orders. Does it ask the same root question?

In [6]:
from sklearn.tree import DecisionTreeClassifier, export_text
X, y = orders[["rush", "rain", "coupon"]], orders.late
tree = DecisionTreeClassifier(criterion="entropy", random_state=0).fit(X, y)
print(export_text(tree, feature_names=list(X.columns)))
assert X.columns[tree.tree_.feature[0]] == "rush" and tree.score(X, y) == 1.0; print("OK")

|--- rush <= 0.50
|   |--- rain <= 0.50
|   |   |--- class: 1
|   |--- rain >  0.50
|   |   |--- class: 0
|--- rush >  0.50
|   |--- class: 1

OK


## Task 5: One-hot encoding
A vehicle column has three values. Turn it into three yes/no columns.

In [7]:
vehicle = pd.DataFrame({"vehicle": ["bike", "scooter", "scooter", "bike", "van"]})
onehot = pd.get_dummies(vehicle, columns=["vehicle"], dtype=int)
print(onehot)
assert list(onehot.columns) == ["vehicle_bike", "vehicle_scooter", "vehicle_van"] and (onehot.sum(axis=1) == 1).all(); print("OK")

   vehicle_bike  vehicle_scooter  vehicle_van
0             1                0            0
1             0                1            0
2             0                1            0
3             1                0            0
4             0                0            1
OK


## Task 6: Sampling with replacement
Draw 10 orders from orders 1 to 10 with replacement. How many different orders appear? Repeat 10,000 times and compare with the theory `1 - 0.9**10`.

In [8]:
rng = np.random.default_rng(0)
draw = rng.integers(1, 11, size=10)            # 10 draws, numbers 1 to 10
print(sorted(draw), "distinct:", len(set(draw)))

distinct = [len(set(rng.integers(1, 11, size=10))) for _ in range(10000)]
print("average distinct share:", np.mean(distinct)/10, " theory:", 1 - 0.9**10)
assert abs(np.mean(distinct)/10 - (1 - 0.9**10)) < 0.01; print("OK")

[np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(9), np.int64(9)] distinct: 7
average distinct share: 0.6525299999999999  theory: 0.6513215599
OK


## Task 7: Tree vs random forest vs XGBoost on delivery data
Invented data with 600 training orders and 3000 test orders. Fit the three models and compare test accuracy.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def make(n, seed):
    r = np.random.default_rng(seed)
    d = pd.DataFrame({"distance": r.uniform(0.5, 10, n), "rush": r.integers(0, 2, n), "rain": r.integers(0, 2, n),
                      "prep_min": r.uniform(5, 30, n), "rider_exp": r.uniform(0, 5, n), "coupon": r.integers(0, 2, n),
                      "vehicle": r.integers(0, 3, n)})
    z = (-3.2 + 0.35*d.distance + 1.4*d.rush*(d.distance > 5) + 1.1*d.rain*(d.vehicle == 0)
         + 0.06*d.prep_min - 0.25*d.rider_exp + r.normal(0, 0.6, n))
    d["late"] = (z > 0).astype(int)
    return pd.get_dummies(d, columns=["vehicle"], dtype=int)      # one-hot the vehicle

train, test = make(600, seed=1), make(3000, seed=1001)
Xtr, ytr = train.drop(columns="late"), train.late
Xte, yte = test.drop(columns="late"), test.late

models = {
    "one tree":      DecisionTreeClassifier(random_state=0),
    "random forest": RandomForestClassifier(n_estimators=200, random_state=0),
    "XGBoost":       XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=0),
}
scores = {}
for name, model in models.items():
    model.fit(Xtr, ytr)
    scores[name] = model.score(Xte, yte)
    print(f"{name:14s} test accuracy {scores[name]:.3f}")
assert scores["random forest"] > scores["one tree"] and scores["XGBoost"] > scores["one tree"]; print("OK: an ensemble beats a single tree")

one tree       test accuracy 0.845
random forest  test accuracy 0.889
XGBoost        test accuracy 0.894
OK: an ensemble beats a single tree


## Task 8: XGBoost for numbers (regression) and a look at which columns mattered
Use `XGBRegressor` to predict a number, then print the feature importances of the forest.

In [10]:
from xgboost import XGBRegressor
rng = np.random.default_rng(5)
dist = rng.uniform(0.5, 10, 400).reshape(-1, 1)
minutes = 12 + 14*np.sqrt(dist[:, 0]) + rng.normal(0, 2.5, 400)          # same idea as Lecture 25
reg = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=2, random_state=0)
reg.fit(dist[:300], minutes[:300])
pred = reg.predict(dist[300:])
rmse = float(np.sqrt(np.mean((pred - minutes[300:])**2)))
print("test RMSE (minutes):", round(rmse, 2), "(the noise floor is 2.5)")
assert rmse < 3.2; print("OK")

imp = pd.Series(models["random forest"].feature_importances_, index=Xtr.columns).sort_values(ascending=False)
print(imp.round(3))

test RMSE (minutes): 2.51 (the noise floor is 2.5)
OK
distance     0.568
prep_min     0.141
rider_exp    0.137
rush         0.058
rain         0.032
vehicle_0    0.026
coupon       0.015
vehicle_1    0.014
vehicle_2    0.010
dtype: float64


## Task 9: Boosting by hand (optional)
Six orders: distance 1 to 6 km, minutes `[20, 24, 31, 38, 41, 52]`. Start with the average, then add half of a one-question tree fitted to the errors, three times. Print the error after each round.

In [11]:
from sklearn.tree import DecisionTreeRegressor
x = np.array([[1], [2], [3], [4], [5], [6]]); y = np.array([20., 24, 31, 38, 41, 52])
pred = np.full(6, y.mean())
mse = [np.mean((y - pred)**2)]
for round_ in range(3):
    residual = y - pred
    stump = DecisionTreeRegressor(max_depth=1).fit(x, residual)       # fit to the errors
    pred = pred + 0.5*stump.predict(x)
    mse.append(np.mean((y - pred)**2))
print([round(m, 1) for m in mse])
assert [round(m, 1) for m in mse] == [115.6, 50.2, 24.9, 9.7]; print("OK")

[np.float64(115.6), np.float64(50.2), np.float64(24.9), np.float64(9.7)]
OK


## Task 10: Your own project (written answer)
Is your data a table, or images, sound or text? Which model family would you try first (tree ensemble or neural network), and which measure on the cross-validation set would you use to decide?

In [12]:
my_answer = "My data is tabular (a table). I will try a tree ensemble model first (such as XGBoost or RandomForest) because they perform exceptionally well on tabular data without extensive scaling. I will use F1-score or AUC-ROC on the cross-validation set to make decisions depending on the class balance."
print(my_answer)

My data is tabular (a table). I will try a tree ensemble model first (such as XGBoost or RandomForest) because they perform exceptionally well on tabular data without extensive scaling. I will use F1-score or AUC-ROC on the cross-validation set to make decisions depending on the class balance.


## Wrap-up
- Entropy measures mixed-up nodes; information gain = entropy before minus weighted entropy after.
- A tree grows by recursion; one-hot encoding turns a k-valued column into k yes/no columns.
- Sampling with replacement gives diverse training sets for a forest; XGBoost builds trees in turn to fix errors.
- Tables: try tree ensembles first. Images, sound and text: neural networks.